# Forge Specialized AI - Fine-Tuning Pipeline

This notebook uses [Unsloth](https://github.com/unslothai/unsloth) to fine-tune `Qwen/Qwen2.5-0.5B-Instruct` on the `training_data.jsonl` dataset generated by your Forge macro recorder.

**Instructions:**
1. Upload your `training_data.jsonl` file to the Colab environment.
2. Run all cells.
3. Download the generated `.gguf` file at the end.
4. Place the `.gguf` file in your Forge `models/` directory and set `FORGE_MODEL` to its path.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192 # Support long UI Contexts
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
import json
from datasets import Dataset

forge_prompt = """<|im_start|>system
You are a Windows PC automation agent. You output ONE JSON action for the next step. No text, just JSON.
Available action types: click_element, type, key, sleep, done.
<|im_end|>
<|im_start|>user
TASK: {}

DONE SO FAR:
None yet

SCREEN ELEMENTS:
{}
<|im_end|>
<|im_start|>assistant
{}"""

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    uias = examples["uia"]
    completions = examples["completion"]
    texts = []
    for p, u, c in zip(prompts, uias, completions):
        # The macro recorder saves a list of actions. We can train it to output the first action, 
        # or if we want, we can train it to output the whole array if we change the Forge planner.
        # For now, let's train it to predict the FIRST action to get started.
        first_action = c[0] if len(c) > 0 else {"type":"done"}
        action_json = json.dumps(first_action)
        text = forge_prompt.format(p, u, action_json) + "<|im_end|>"
        texts.append(text)
    return { "text" : texts }

def load_dataset_from_jsonl(file_path):
    with open(file_path, 'r') as f:
        data = [json.loads(line) for line in f]
    return Dataset.from_list(data)

dataset = load_dataset_from_jsonl("training_data.jsonl")
dataset = dataset.map(formatting_prompts_func, batched = True,)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Export to GGUF
model.save_pretrained_gguf("forge_model", tokenizer, quantization_method = "q4_k_m")